# ocr

> Read a page with the PP-OCRv6 medium detector and recognizer.


In [ ]:
#| default_exp ocr


#| export
Pages are resized and boxes are read off the detector map the same way as PaddleOCR: keep the aspect ratio, then take DB contours, a minimum-area rectangle, and an unclip. Recognition is CTC over the LightSVTR sequence. Compilation stays inside `main`, so the geometry below can be read on its own.


In [ ]:
#| hide
from nbdev import show_doc


In [ ]:
#| export
from __future__ import annotations

import argparse
import math
import sys
from pathlib import Path

from solid_lens.core import repo_root


In [ ]:
#| export
ROOT = repo_root()


## Detector size


These are the PP-OCRv6 medium overrides from PaddleOCR's OCR pipeline. The checkpoint yaml is looser (0.2 / 0.45 / 1.4). The pipeline that produces the boxes uses the numbers below.


In [ ]:
#| export
DET_LIMIT_SIDE_LEN = 64
DET_LIMIT_TYPE = "min"
DET_MAX_SIDE_LIMIT = 4000
DET_THRESH = 0.3
DET_BOX_THRESH = 0.6
DET_UNCLIP_RATIO = 1.5
DET_MAX_CANDIDATES = 3000
DET_MIN_SIZE = 3


In [ ]:
#| export
def detection_hw(
    height: int, # Page height, in pixels
    width: int, # Page width, in pixels
    limit_side_len: int = DET_LIMIT_SIDE_LEN, # Side the limit type compares against
    limit_type: str = DET_LIMIT_TYPE, # `min`, `max`, or `resize_long`
    max_side_limit: int = DET_MAX_SIDE_LIMIT,
) -> tuple[int, int]:
    "Detector height and width. Same rule as PaddleOCR `DetResizeForTest` type 0."
    if limit_type == "max":
        ratio = float(limit_side_len) / max(height, width) if max(height, width) > limit_side_len else 1.0
    elif limit_type == "min":
        ratio = float(limit_side_len) / min(height, width) if min(height, width) < limit_side_len else 1.0
    elif limit_type == "resize_long":
        ratio = float(limit_side_len) / max(height, width)
    else:
        raise ValueError(f"unsupported limit type {limit_type}")
    resize_h = int(height * ratio)
    resize_w = int(width * ratio)
    if max(resize_h, resize_w) > max_side_limit:
        fitted = float(max_side_limit) / max(resize_h, resize_w)
        resize_h = int(resize_h * fitted)
        resize_w = int(resize_w * fitted)
    resize_h = max(int(round(resize_h / 32) * 32), 32)
    resize_w = max(int(round(resize_w / 32) * 32), 32)
    return resize_h, resize_w


`limit_type` is `min` and `limit_side_len` is 64, so a page whose short side is already at least 64 is not shrunk. Both sides are then rounded to a multiple of 32, and nothing is allowed below 32. `round` is bankers' rounding, so a half step lands on the even multiple.


In [ ]:
from fastcore.test import test_eq

test_eq(detection_hw(100, 200), (96, 192))
test_eq(detection_hw(40, 80), (64, 128))
test_eq(detection_hw(100, 400, limit_side_len=64, limit_type="resize_long"), (32, 64))
detection_hw(100, 200)


(96, 192)

## CTC


Class 0 is the blank. Repeated labels collapse, and blanks separate the repeats that should survive.


In [ ]:
#| export
def _ctc_decode(ids: list[int], charset: list[str]) -> str:
    text = []
    previous = -1
    for index in ids:
        if index != 0 and index != previous:
            text.append(charset[index - 1])
        previous = index
    return "".join(text).strip()


In [ ]:
from fastcore.test import test_eq

test_eq(_ctc_decode([0, 1, 1, 0, 2, 2, 3], ["a", "b", "c"]), "abc")
test_eq(_ctc_decode([1, 1], ["a"]), "a")
_ctc_decode([0, 1, 0, 1], ["a", "b"])


'aa'

## Images


In [ ]:
#| export
import cv2
import numpy as np
import pyclipper
from PIL import Image


In [ ]:
#| export
DET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
DET_STD = np.array([0.229, 0.224, 0.225], np.float32)


Detector normalization is ImageNet mean and std on a BGR image. The recognizer uses a different scale, `(x / 255 - 0.5) / 0.5`, applied later per crop.


In [ ]:
#| export
def load_charset(path: Path, *, add_space: bool = False) -> list[str]:
    chars = path.read_text().splitlines()
    if add_space:
        chars.append(" ")
    return chars


In [ ]:
#| export
def load_image(path: Path) -> np.ndarray:
    "Return an RGB uint8 image."
    with Image.open(path) as image:
        return np.ascontiguousarray(image.convert("RGB"))


In [ ]:
#| export
def resize_hwc(image: np.ndarray, out_h: int, out_w: int) -> np.ndarray:
    src_h, src_w = image.shape[:2]
    if (src_h, src_w) == (out_h, out_w):
        return image
    y = (np.arange(out_h, dtype=np.float32) + 0.5) * src_h / out_h - 0.5
    x = (np.arange(out_w, dtype=np.float32) + 0.5) * src_w / out_w - 0.5
    y = np.clip(y, 0, src_h - 1)
    x = np.clip(x, 0, src_w - 1)
    y0 = np.floor(y).astype(np.int32)
    x0 = np.floor(x).astype(np.int32)
    y1 = np.clip(y0 + 1, 0, src_h - 1)
    x1 = np.clip(x0 + 1, 0, src_w - 1)
    wy = (y - y0).astype(np.float32)[:, None, None]
    wx = (x - x0).astype(np.float32)[None, :, None]
    top = image[y0][:, x0] * (1 - wx) + image[y0][:, x1] * wx
    bot = image[y1][:, x0] * (1 - wx) + image[y1][:, x1] * wx
    return (top * (1 - wy) + bot * wy).astype(image.dtype)


This is bilinear resampling with the pixel center at the half coordinate, which is what the recognizer crops use. An image that is already the requested size is returned unchanged.


In [ ]:
import numpy as np
from fastcore.test import test_eq

image = np.arange(12, dtype=np.uint8).reshape(3, 4, 1).repeat(3, axis=2)
same = resize_hwc(image, 3, 4)
test_eq(same.shape, image.shape)
test_eq(bool(same is image), True)
small = resize_hwc(image, 1, 1)
test_eq(small.shape, (1, 1, 3))
small


array([[[5, 5, 5]]], dtype=uint8)

In [ ]:
#| export
def det_input(image_rgb: np.ndarray, out_h: int, out_w: int) -> np.ndarray:
    "Resize like Paddle and normalize to a BGR NCHW tensor."
    resized = cv2.resize(image_rgb, (out_w, out_h))
    bgr = resized[:, :, ::-1].astype(np.float32)
    normalized = (bgr / 255.0 - DET_MEAN) / DET_STD
    return np.ascontiguousarray(np.transpose(normalized, (2, 0, 1))[None], dtype=np.float32)


In [ ]:
import numpy as np
from fastcore.test import test_eq

page = np.zeros((8, 16, 3), np.uint8)
tensor = det_input(page, 32, 32)
test_eq(tensor.shape, (1, 3, 32, 32))
test_eq(tensor.dtype, np.float32)
tensor.shape


(1, 3, 32, 32)

## Boxes


In [ ]:
#| export
def _mini_boxes(contour: np.ndarray) -> tuple[list[list[float]], float]:
    "Four corners of the minimum-area rectangle, and its shorter side."
    rect = cv2.minAreaRect(contour)
    points = sorted(cv2.boxPoints(rect).tolist(), key=lambda point: point[0])
    index_1, index_4 = (0, 1) if points[1][1] > points[0][1] else (1, 0)
    index_2, index_3 = (2, 3) if points[3][1] > points[2][1] else (3, 2)
    box = [points[index_1], points[index_2], points[index_3], points[index_4]]
    return box, min(rect[1])


In [ ]:
#| export
def _box_score_fast(bitmap: np.ndarray, box: np.ndarray) -> float:
    "Mean detector score inside the quadrilateral."
    height, width = bitmap.shape[:2]
    xmin = max(0, min(math.floor(float(box[:, 0].min())), width - 1))
    xmax = max(0, min(math.ceil(float(box[:, 0].max())), width - 1))
    ymin = max(0, min(math.floor(float(box[:, 1].min())), height - 1))
    ymax = max(0, min(math.ceil(float(box[:, 1].max())), height - 1))
    mask = np.zeros((ymax - ymin + 1, xmax - xmin + 1), np.uint8)
    shifted = box.copy()
    shifted[:, 0] -= xmin
    shifted[:, 1] -= ymin
    cv2.fillPoly(mask, shifted.reshape(1, -1, 2).astype(np.int32), 1)
    return float(cv2.mean(bitmap[ymin : ymax + 1, xmin : xmax + 1], mask)[0])


In [ ]:
#| export
def _unclip(box: np.ndarray, unclip_ratio: float) -> np.ndarray:
    "Offset the polygon by `area * ratio / perimeter`, as `DBPostProcess` does."
    area = cv2.contourArea(box)
    length = cv2.arcLength(box, True)
    if length <= 0:
        return np.zeros((0, 2), np.float32)
    offset = pyclipper.PyclipperOffset()
    offset.AddPath(box, pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
    expanded = offset.Execute(area * unclip_ratio / length)
    if not expanded:
        return np.zeros((0, 2), np.float32)
    try:
        return np.array(expanded)
    except ValueError:
        return np.array(expanded[0])


The unclip distance is the DB post-process offset: area times the ratio, divided by the perimeter. Round joins match PaddleOCR.


In [ ]:
#| export
def boxes_from_bitmap(
    prob: np.ndarray,
    dest_width: int,
    dest_height: int,
    *,
    thresh: float = DET_THRESH,
    box_thresh: float = DET_BOX_THRESH,
    unclip_ratio: float = DET_UNCLIP_RATIO,
    max_candidates: int = DET_MAX_CANDIDATES,
) -> list[tuple[int, int, int, int, float]]:
    "PaddleOCR `DBPostProcess` quads, returned as axis-aligned boxes."
    bitmap = prob > thresh
    height, width = bitmap.shape
    width_scale = dest_width / width
    height_scale = dest_height / height
    found = cv2.findContours((bitmap.astype(np.uint8) * 255), cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = found[0] if len(found) == 2 else found[1]
    boxes: list[tuple[int, int, int, int, float]] = []
    for contour in contours[:max_candidates]:
        points, short_side = _mini_boxes(contour)
        if short_side < DET_MIN_SIZE:
            continue
        points_arr = np.asarray(points, dtype=np.float32)
        score = _box_score_fast(prob, points_arr)
        if score < box_thresh:
            continue
        expanded = _unclip(points_arr, unclip_ratio)
        if expanded.size == 0:
            continue
        expanded, short_side = _mini_boxes(expanded.reshape(-1, 1, 2))
        if short_side < DET_MIN_SIZE + 2:
            continue
        quad = np.asarray(expanded, dtype=np.float64)
        quad[:, 0] = np.clip(np.round(quad[:, 0] * width_scale), 0, dest_width)
        quad[:, 1] = np.clip(np.round(quad[:, 1] * height_scale), 0, dest_height)
        left = int(quad[:, 0].min())
        top = int(quad[:, 1].min())
        right = int(quad[:, 0].max())
        bottom = int(quad[:, 1].max())
        if right <= left or bottom <= top:
            continue
        boxes.append((left, top, right, bottom, score))
    boxes.sort(key=lambda box: (box[1], box[0]))
    return boxes


Coordinates come back in the original image. The recognizer crops that rectangle. Boxes are sorted top to bottom, then left to right. A contour whose short side is under 3 pixels, or whose mean score is under 0.6, is dropped. After the unclip the short side must clear 5.


In [ ]:
import numpy as np
from fastcore.test import test_eq

prob = np.zeros((64, 64), np.float32)
prob[20:36, 8:48] = 0.95
boxes = boxes_from_bitmap(prob, 64, 64)
test_eq(len(boxes), 1)
left, top, right, bottom, score = boxes[0]
test_eq(score > 0.6, True)
test_eq(left < 8, True)
test_eq(right > 48, True)
boxes[0]


(0, 12, 55, 43, 0.949999988079071)

## Recognition


In [ ]:
#| export
def _recognize_ids(model, image_rgb: np.ndarray) -> list[int]:
    "Run one crop that already fits the 320-wide recognizer."
    height, width = image_rgb.shape[:2]
    resized_w = int(np.ceil(48 * width / height))
    resized_w = max(8, min(320, resized_w))
    resized = resize_hwc(image_rgb, 48, resized_w).astype(np.float32)
    bgr = resized[:, :, ::-1]
    normalized = np.transpose((bgr / 255.0 - 0.5) / 0.5, (2, 0, 1))
    tensor = np.zeros((1, 3, 48, 320), np.float32)
    tensor[0, :, :, :resized_w] = normalized
    tensor = np.ascontiguousarray(tensor)
    probs = model.execute(tensor)[0].to_numpy()[0]
    # The recognizer downsamples width by 8, so padding past resized_w is blank.
    valid = max(1, resized_w // 8)
    return [int(index) for index in probs.argmax(axis=-1)[:valid]]


In [ ]:
#| export
def recognize_crop(model, image_rgb: np.ndarray, charset: list[str]) -> str:
    height, width = image_rgb.shape[:2]
    if height < 2 or width < 2:
        return ""
    fitted = int(np.ceil(48 * width / height))
    if fitted <= 320:
        return _ctc_decode(_recognize_ids(model, image_rgb), charset)
    # A long line squashed into 320 columns loses characters. Slide a window
    # whose kept timesteps abut, and drop the edges where the crop has no context.
    drop = 5
    stride_net = (40 - 2 * drop) * 8
    window = max(8, int(round(320 * height / 48)))
    stride = max(4, int(round(stride_net * height / 48)))
    merged: list[int] = []
    x = 0
    first = True
    while x < width:
        piece = image_rgb[:, x : min(width, x + window)]
        seq = _recognize_ids(model, piece)
        last = x + window >= width
        if first and last:
            use = seq
        elif first:
            use = seq[:-drop] if len(seq) > drop else seq
        elif last:
            use = seq[drop:] if len(seq) > drop else seq
        elif len(seq) > 2 * drop:
            use = seq[drop:-drop]
        else:
            use = seq
        merged.extend(use)
        if last:
            break
        x += stride
        first = False
    return _ctc_decode(merged, charset)


A line that fits in 320 columns is one forward pass. A longer line is a sliding window. Five timesteps are dropped on each interior edge, where the crop has no context, and the stride is chosen so the kept timesteps abut. The network's width stride is 8.


In [ ]:
#| export
def detect_boxes(image_rgb: np.ndarray, det_model) -> list[tuple[int, int, int, int, float]]:
    "Text boxes in original pixels, from one forward pass at the PaddleOCR det size."
    height, width = image_rgb.shape[:2]
    out_h, out_w = detection_hw(height, width)
    prob = det_model.execute(det_input(image_rgb, out_h, out_w))[0].to_numpy()[0, 0]
    return boxes_from_bitmap(prob, width, height)


In [ ]:
#| export
def read_page(
    image_rgb: np.ndarray,
    det_model,
    rec_model,
    charset: list[str],
) -> list[tuple[str, float, int, int, int, int]]:
    lines = []
    for left, top, right, bottom, score in detect_boxes(image_rgb, det_model):
        text = recognize_crop(rec_model, image_rgb[top:bottom, left:right], charset)
        if text:
            lines.append((text, score, left, top, right, bottom))
    return lines


`read_page` drops crops the recognizer turns into an empty string. The score on each line is the detector score of that box, not a recognition probability.


In [ ]:
#| export
def draw_boxes(image_rgb: np.ndarray, lines: list[tuple[str, float, int, int, int, int]]) -> np.ndarray:
    "Paint a green rectangle around each recognized line."
    canvas = image_rgb.copy()
    height, width = canvas.shape[:2]
    color = np.array([32, 220, 80], np.uint8)
    for _, _, left, top, right, bottom in lines:
        x0 = int(np.clip(left, 0, width - 1))
        y0 = int(np.clip(top, 0, height - 1))
        x1 = int(np.clip(right, x0 + 1, width))
        y1 = int(np.clip(bottom, y0 + 1, height))
        thickness = 2
        canvas[y0 : y0 + thickness, x0:x1] = color
        canvas[y1 - thickness : y1, x0:x1] = color
        canvas[y0:y1, x0 : x0 + thickness] = color
        canvas[y0:y1, x1 - thickness : x1] = color
    return canvas


In [ ]:
#| export
def save_rgb(path: Path, image_rgb: np.ndarray) -> None:
    Image.fromarray(np.ascontiguousarray(image_rgb), mode="RGB").save(path)


## Command


In [ ]:
#| export
def main() -> None:
    from solid_lens.model import compile_detector, compile_recognizer

    parser = argparse.ArgumentParser(description="Run PP-OCRv6 medium OCR with MAX")
    parser.add_argument("image", type=Path)
    parser.add_argument("--output", type=Path, help="image with boxes drawn; defaults to <image>.boxes.png")
    args = parser.parse_args()
    image = load_image(args.image)
    det_h, det_w = detection_hw(*image.shape[:2])
    charset = load_charset(ROOT / "models" / "ppocrv6_dict.txt", add_space=True)
    det_cache = ROOT / "models" / f"native_medium_det_{det_h}x{det_w}.mef"
    rec_cache = ROOT / "models" / "native_medium_rec_48x320.mef"
    print(
        f"{'loading cached' if any(det_cache.glob('*.mef')) else 'compiling'} PP-OCRv6 medium detector {det_h}x{det_w}",
        file=sys.stderr,
    )
    detector = compile_detector(height=det_h, width=det_w, cache_dir=det_cache)
    print(
        "loading cached recognizer" if any(rec_cache.glob("*.mef")) else "compiling PP-OCRv6 medium recognizer",
        file=sys.stderr,
    )
    recognizer = compile_recognizer(cache_dir=rec_cache)
    lines = read_page(image, detector, recognizer, charset)
    output = args.output or args.image.with_name(f"{args.image.stem}.boxes.png")
    save_rgb(output, draw_boxes(image, lines))
    print(f"wrote {output}", file=sys.stderr)
    for text, score, *_ in lines:
        print(f"{score:.2f}  {text}")


In [ ]:
#| export
#| notest
if __name__ == "__main__":
    main()
